In [1]:
batch_size = 4
n_epochs = 200
lr = 1e-3

In [2]:
from sklearn.datasets import load_breast_cancer

dataset = load_breast_cancer()

X = dataset.data
Y = dataset.target
input_size = X.shape[-1]


print(X.shape)  # (569, 30)
print(Y.shape)  # (569,)

(569, 30)
(569,)


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train).T
X_test = scaler.transform(X_test).T

Y_train = Y_train.reshape(-1, 1)
Y_test = Y_test.reshape(-1, 1)

In [4]:
import math

n_test_samples = X_test.shape[-1]
n_test_batches = X_test.shape[-1]
n_train_samples = X_train.shape[-1]
n_train_batches = math.ceil(n_train_samples / batch_size)

In [5]:
import numpy as np


class Sigmoid:
    Y_hat: np.ndarray
    def forward(self, z, training: bool=True):
        Y_hat = 1 / (1 + np.exp(-z))
        if training:
            self.Y_hat = Y_hat

        return Y_hat
    
    def backward(self, dY_hat: np.ndarray):
        # dLoss/dY_hat * dY_hat/dz
        return dY_hat * (self.Y_hat * (1 - self.Y_hat))

class BCELoss:
    def __init__(self):
        self.eps = 1e-7
    
    def forward(self, Y_hat, Y, training: bool=True):
        if training:
            self.Y_hat = Y_hat
            self.Y = Y
        
        Y_hat_ = np.clip(Y_hat, self.eps, 1 - self.eps)
    
        return -(
            Y * np.log(Y_hat_)
            + (1- Y) * np.log(1 - Y_hat_)
        )
    
    def backward(self):
        Y_hat_ = np.clip(self.Y_hat, self.eps, 1 - self.eps)
        self.dY_hat = -self.Y / Y_hat_ + (1- self.Y) / (1 - Y_hat_)
        return self.dY_hat

class LRModel:
    W: np.ndarray 
    b: float
    Y_hat: np.ndarray
    dW: np.ndarray
    db: float
    def __init__(self, input_size, batch_size, activation, criterion, lr):
        self.input_size = input_size
        self.batch_size = batch_size
        self.activation = activation
        self.criterion = criterion
        self.lr = lr

        self.W = np.zeros((self.input_size, 1))
        self.b = 0
        
        pass
    
    def forward(self, X: np.ndarray, Y: np.ndarray, training: bool = True):
        if training:
            self.X = X
            self.Y = Y

        # get prediction
        _, Y_hat = self.predict(X, training=True)
        
        # calculate loss
        loss_batch = self.criterion.forward(
            Y_hat, Y, training=True
        )
        loss_value = np.mean(loss_batch)
        
        return loss_value, Y_hat
    
    def predict(self, X, training=False):
        logit = X.T @ self.W + self.b
        Y_hat = self.activation.forward(logit, training=training)

        return logit, Y_hat

    def backward(self):
        dY_hat = self.criterion.backward()
        self.dZ = self.activation.backward(dY_hat)

        m = self.X.shape[1]
        self.dW = (1 / m) * (self.X @ self.dZ)
        self.db = np.mean(self.dZ)
        
        return
    
    def step(self):
        self.W -= self.lr * self.dW
        self.b -= self.lr * self.db
        
        return

class Evaluator:
    def __init__(self):
        self.cm = np.zeros((2, 2))
    
    def reset(self):
        self.cm = np.zeros((2, 2))
        self.metrics = {}

        return
    
    def step(self, Y_hat, Y):
        assert Y_hat.shape == Y.shape, "Predict and Labels's shape differ"
        cm_step = self.compute_cm(Y_hat, Y)
        self.cm += cm_step

        return
    
    def get_metrics(self):
        [[tn, fp], [fn, tp]] = self.cm
        acc = self.accuracy(tn, fp, fn, tp)
        precision = self.precision(tn, fp, fn, tp)
        recall = self.recall(tn, fp, fn, tp)
        spec = self.specificity(tn, fp, fn, tp)
        f1 = self.f1_score(tn, fp, fn, tp)
        
        self.metrics = {
            'cm': self.cm,
            'acc': acc,
            'precision': precision,
            'recall': recall,
            'spec': spec,
            'f1': f1,
        }
        
        return self.metrics

    def get_confusion_matrix(self):
        return self.cm
    
    @staticmethod
    def accuracy(tn, fp, fn, tp):
        return (tn + tp) / (tn + fp + fn + tp)
    
    @staticmethod
    def precision(tn, fp, fn, tp):
        return tp / (tp + fp)
    
    @staticmethod
    def recall(tn, fp, fn, tp):
        return tp / (tp + fn)
    
    @staticmethod
    def specificity(tn, fp, fn, tp):
        # this is the recall for 0's
        return tn / (tn + fp) 
    
    @staticmethod
    def f1_score(tn, fp, fn, tp):
        return 2 * tp / (2 * tp + fp + fn)
    
    @staticmethod
    def compute_cm( Y_hat, Y):
        tp = (Y_hat * Y).sum()
        tn = ((1 - Y_hat) * (1 - Y)).sum()
        fp = (Y_hat * (1 - Y)).sum()
        fn = ((1 - Y_hat) * Y).sum()
        
        return np.array(
            [[tn, fp],
             [fn, tp]]
        )
        

In [6]:
header = (
    f"{'Epoch':<8}"
    f"{'Mode':<8}"
    f"{'N Samples':<10}"
    f"{'Loss':<10}"
    f"{'Acc':<10}"
    f"{'Prec':<10}"
    f"{'Recall':<10}"
    f"{'Spec':<10}"
    f"{'F1':<10}"
    "Progress"
)

def format_row(epoch, mode, n_samples, loss, metrics=None, indent=0):
    metrics = metrics or {}
    metric_keys = ["acc", "precision", "recall", "spec", "f1"]

    def fmt(key):
        value = metrics.get(key)
        return f"{value:<10.4f}" if value is not None else f"{'-':<10}"

    return " " * indent + "".join([
        f"{epoch:<8}",
        f"{mode:<8}",
        f"{n_samples:<10}",
        f"{loss:<10.6f}",
        *[fmt(key) for key in metric_keys]
    ])

In [8]:
from tqdm.std import tqdm


criterion = BCELoss()
activation = Sigmoid()
train_evaluator = Evaluator()
test_evaluator = Evaluator()

model = LRModel(
    input_size, batch_size, activation, criterion, lr
)

print(header)
print("-" * 120)

for epoch in range(n_epochs):
    pbar = tqdm(
        total=n_train_batches,
        desc=format_row(epoch + 1, "Train", n_train_samples, 0.0),
        leave=True,
        bar_format="{desc} {bar:20} {percentage:3.0f}%"
    )

    # ------------------------------ TRAIN STEP -------------------------------
    train_loss = 0
    train_evaluator.reset()
    
    for batch_i, i in enumerate(range(0, n_train_samples, batch_size)):
        X_batch = X_train[:, i:(i + batch_size)]
        Y_batch = Y_train[i:(i + batch_size), :]
        m = X_batch.shape[1]
        
        batch_loss, Y_hat_batch = model.forward(X_batch, Y_batch)  # batch loss
        train_loss = (i * train_loss + m * batch_loss) / (i + m)  # epoch loss

        model.backward()
        model.step()
        train_evaluator.step(Y_hat_batch, Y_batch)

        # report
        pbar.set_description_str(
            format_row(epoch + 1, "Train", n_train_samples, train_loss),
            refresh=False
        )
        pbar.update(1)

    train_metrics = train_evaluator.get_metrics()
    pbar.set_description_str(
        format_row(epoch + 1, "Train", n_train_samples,
                   train_loss, train_metrics),
        refresh=True
    )

    pbar.close()

    # ---------------------------- TEST STEP ----------------------------------
    pbar = tqdm(
        total=n_test_batches,
        desc=format_row(epoch + 1, "Test", n_test_samples, 0.0),
        leave=True,
        bar_format="{desc} {bar:20} {percentage:3.0f}%"
    )

    test_loss = 0
    test_evaluator.reset()

    for i in range(n_test_samples):
        X_batch = X_test[:, i:(i+1)]
        Y_batch = Y_test[i:(i+1), :]
        
        batch_loss, Y_hat_batch = model.forward(
            X_batch, Y_batch, training=False
        )
        test_loss = (test_loss * i + batch_loss) / (i + 1)

        test_evaluator.step(Y_hat_batch, Y_batch)
        
        # report
        pbar.set_description_str(
            format_row(epoch + 1, "Test", n_test_samples, test_loss),
            refresh=False
        )
        pbar.update(1)
    
    test_metrics = test_evaluator.get_metrics()
    pbar.set_description_str(
        format_row("", "└─ Test", n_test_samples,
                   test_loss, test_metrics),
        refresh=True
    )
    pbar.close()
    

Epoch   Mode    N Samples Loss      Acc       Prec      Recall    Spec      F1        Progress
------------------------------------------------------------------------------------------------------------------------


1       Train   455       0.600885  0.5504    0.6760    0.5418    0.5647    0.6015     ████████████████████ 100%
        └─ Test 114       0.535209  0.5895    0.7222    0.5689    0.6249    0.6364     ████████████████████ 100%
2       Train   455       0.476491  0.6270    0.7487    0.6088    0.6574    0.6715     ████████████████████ 100%
        └─ Test 114       0.450346  0.6455    0.7764    0.6161    0.6959    0.6870     ████████████████████ 100%
3       Train   455       0.404649  0.6766    0.7921    0.6558    0.7114    0.7176     ████████████████████ 100%
        └─ Test 114       0.397081  0.6832    0.8109    0.6500    0.7402    0.7216     ████████████████████ 100%
4       Train   455       0.357560  0.7111    0.8199    0.6905    0.7457    0.7496     ████████████████████ 100%
        └─ Test 114       0.360108  0.7105    0.8345    0.6757    0.7703    0.7467     ████████████████████ 100%
5       Train   455       0.323971  0.7366    0.8390    0.7172    0.7692    0.7733     █████████